<a href="https://colab.research.google.com/github/MaisZiwana/TCF7L2-MENA-Pharmacogenomics/blob/main/TCF7L2_Analyisis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from google.colab import files

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

freq = pd.read_excel(file_name, sheet_name="Sheet3")
n = pd.read_excel(file_name, sheet_name="Sheet4")

df = freq.merge(n, on="rsID")

df = df.rename(columns={
    "Middle Eastern AF":"ME_AF",
    "European AF":"EUR_AF",
    "Middle Eastern AN":"ME_AN",
    "European AN":"EUR_AN"
})

df["ME_Risk"] = df["ME_AF"] * df["ME_AN"]
df["ME_NonRisk"] = df["ME_AN"] - df["ME_Risk"]

df["EUR_Risk"] = df["EUR_AF"] * df["EUR_AN"]
df["EUR_NonRisk"] = df["EUR_AN"] - df["EUR_Risk"]

chi2_list = []
p_list = []
or_list = []

for _, row in df.iterrows():

    table = [
        [row["ME_Risk"], row["ME_NonRisk"]],
        [row["EUR_Risk"], row["EUR_NonRisk"]]
    ]

    chi2, p, _, _ = chi2_contingency(table)

    OR = (row["ME_Risk"] * row["EUR_NonRisk"]) / (
         row["EUR_Risk"] * row["ME_NonRisk"]
    )

    chi2_list.append(chi2)
    p_list.append(p)
    or_list.append(OR)

df["Chi2"] = chi2_list
df["p_value"] = p_list
df["Odds_Ratio"] = or_list

output_file = "TCF7L2_Final_ChiSquare.xlsx"
df.to_excel(output_file, index=False)

files.download(output_file)

print(df)

Saving Book (2).xlsx to Book (2).xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

         rsID Risk Allele  Global AF   ME_AF  EUR_AF  \
0   rs7903146           T     0.2740  0.4048  0.2919   
1   rs4506565           T     0.3351  0.4184  0.3150   
2   rs7901695           C     0.3337  0.4150  0.3121   
3  rs12243326           C     0.2673  0.3367  0.2764   
4  rs34872471           C     0.2913  0.4150  0.2938   

   Enrichment Ratio (ME/EUR)  Difference (MEAF-EURAF)  ME_AN  EUR_AN  \
0                   1.386776                   0.1129    294   67952   
1                   1.328254                   0.1034    294   67976   
2                   1.329702                   0.1029    294   67972   
3                   1.218162                   0.0603    294   67982   
4                   1.412526                   0.1212    294   67954   

   Chi-square (p-value)   ME_Risk  ME_NonRisk    EUR_Risk  EUR_NonRisk  \
0                   NaN  119.0112    174.9888  19835.1888   48116.8112   
1                   NaN  123.0096    170.9904  21412.4400   46563.5600   
2       

In [ ]:
df["Significance"] = np.where(
    df["p_value"] < 0.001,
    "Highly significant",
    "Significant"
)

df.to_excel("TCF7L2_Final_ChiSquare_with_significance.xlsx", index=False)

from google.colab import files
files.download("TCF7L2_Final_ChiSquare_with_significance.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_excel(file_name)

ME_AN = 294 * 2
EUR_AN = 67952 * 2

df["ME_Risk"] = df["ME_AF"] * ME_AN
df["ME_NonRisk"] = ME_AN - df["ME_Risk"]

df["EUR_Risk"] = df["EUR_AF"] * EUR_AN
df["EUR_NonRisk"] = EUR_AN - df["EUR_Risk"]

ci_low = []
ci_high = []

for _, row in df.iterrows():

    OR = row["Odds_Ratio"]

    se = np.sqrt(
        1/row["ME_Risk"] +
        1/row["ME_NonRisk"] +
        1/row["EUR_Risk"] +
        1/row["EUR_NonRisk"]
    )

    low = np.exp(np.log(OR) - 1.96 * se)
    high = np.exp(np.log(OR) + 1.96 * se)

    ci_low.append(low)
    ci_high.append(high)

df["CI_95_low"] = ci_low
df["CI_95_high"] = ci_high

df.to_excel("TCF7L2_Final_with_CI.xlsx", index=False)

files.download("TCF7L2_Final_with_CI.xlsx")

print(df)